In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import os
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from mal_client import MALClient
from anime_data import AnimeDataClient
from anime_recommender import SimilarityRecommender

load_dotenv(PROJECT_ROOT / ".env")

client_id = os.getenv("CLIENT_ID")

In [3]:
# users = set({'Cresherhsm', 'Mevoll', 'Cheuns', 'Haileytokar', 'dragonenjoyer', 'chenamaty', 'monkeydirene', 'MarnikBe', 'ssafin', 'zClaw_Epic', 'ayumix3', 'ikodrmz', 'heemini', 'I_grV', 'Opelo_Stradyon', 'DanDeku', 'Toaster_toaster', 'SaniLani', 'ArceusComplex', 'MubE', 'SuricateVoador', 'Captn_Cook', 'LILITH_OG', 'DoomSlayer_OG', 'Sturmx', 'AkiAki_Akira', 'kaninhoppning', 'kuzyadam', 'N0rth_5tar', 'KyleAxity_'})

Users data

In [4]:
# users_data = {}
# users_scores = {}

# users = set({'Cresherhsm', 'Mevoll', 'Cheuns', 'Haileytokar', 'dragonenjoyer', 'chenamaty', 'monkeydirene', 'MarnikBe', 'ssafin', 'zClaw_Epic', 'ayumix3', 'ikodrmz', 'heemini', 'I_grV', 'Opelo_Stradyon', 'DanDeku', 'Toaster_toaster', 'SaniLani', 'ArceusComplex', 'MubE', 'SuricateVoador', 'Captn_Cook', 'LILITH_OG', 'DoomSlayer_OG', 'Sturmx', 'AkiAki_Akira', 'kaninhoppning', 'kuzyadam', 'N0rth_5tar', 'KyleAxity_'})
# mal_client = MALClient(client_id)
# for user in users:
#     user_data = mal_client.get_user_data(user)
#     users_data[user] = user_data
#     scores = mal_client.get_scores(user_data)
#     users_scores[user] = scores

In [5]:
anime_data_client = AnimeDataClient(
    client_id,
    cache_file=PROJECT_ROOT / "anime_cache.json",
)

In [6]:
anime_data = anime_data_client.get_cache()

Build features

In [7]:
# from anime_features import AnimeFeatureBuilder

# builder = AnimeFeatureBuilder(
#     anime_data,
#     max_tfidf_features=3000,
#     n_svd_components=300
# )

# anime_df = builder.build_features()

# builder.svd_explained_variance

Convert each anime in df to vectors

In [8]:
# recommender = SimilarityRecommender()
# anime_vectors = recommender.create_anime_vectors(anime_df)
# anime_df_scaled = recommender.anime_df_scaled

Get user Data

In [9]:
username = "chekkit"
user_client = MALClient(client_id)

user_data = user_client.get_user_data(username)
user_scores = user_client.get_scores(user_data)

Tune SVD components

In [10]:
from anime_evaluation import HitRateEvaluator, RankingMetricEvaluator
from anime_features import AnimeFeatureBuilder

svd_component_results = []

n_runs = 100
clip_predictions = False
max_features = [2000, 3000, 4000, 5000]
components = [200, 300, 400]
weights_uncertainty = [14]
tuning_top_ks = [5, 10]

for n_feature in max_features:
    for component in components:
        builder = AnimeFeatureBuilder(
            anime_data,
            max_tfidf_features=n_feature,
            n_svd_components=component,
        )

        component_anime_df = builder.build_features()

        recommender = SimilarityRecommender()
        recommender.create_anime_vectors(component_anime_df)
        component_anime_df_scaled = recommender.anime_df_scaled

        hitman = HitRateEvaluator(
            anime_df_scaled=component_anime_df_scaled,
            anime_df=component_anime_df,
            scores=user_scores,
            anime_data_client=anime_data_client,
            anime_data=anime_data,
            builder=builder,
            recommender=recommender,
        )

        (
            bayesian_results,
            bayesian_summary,
            best_bayesian_weights,
            baseline_results,
            baseline_summary,
        ) = hitman.tune_bayesian_uncertainty(
            weights=weights_uncertainty,
            n_runs=n_runs,
            top_ks=tuning_top_ks,
            random_state=42,
            clip_predictions=clip_predictions,
        )

        ranking_evaluator = RankingMetricEvaluator(
            anime_df_scaled=component_anime_df_scaled,
            anime_df=component_anime_df,
            scores=user_scores,
            anime_data_client=anime_data_client,
            anime_data=anime_data,
            builder=builder,
            recommender=recommender,
        )
        _, ranking_summary = ranking_evaluator.tune_bayesian_uncertainty_ranking(
            weights=weights_uncertainty,
            n_runs=n_runs,
            top_ks=tuning_top_ks,
            random_state=42,
            clip_predictions=clip_predictions,
        )

        bayesian_summary = bayesian_summary.rename(
            columns={"uncertainty_weight": "bayesian_uncertainty_weight"}
        )
        ranking_summary = ranking_summary.rename(
            columns={"uncertainty_weight": "bayesian_uncertainty_weight"}
        )

        average_metrics = (
            bayesian_summary
            .merge(
                ranking_summary,
                on=["bayesian_uncertainty_weight", "k"],
                how="left",
            )
            .merge(
                baseline_summary,
                on="k",
                how="left",
            )
        )
        average_metrics["component"] = component
        average_metrics["n_feature"] = n_feature
        average_metrics["clip_predictions"] = clip_predictions
        average_metrics["n_runs"] = n_runs
        average_metrics["svd_explained_variance"] = builder.svd_explained_variance

        svd_component_results.append(average_metrics)

svd_component_summary = (
    pd.concat(svd_component_results, ignore_index=True)
    .sort_values(
        ["k", "avg_precision_at_k", "avg_ndcg_at_k"],
        ascending=[True, False, False],
    )
)

metrics_path = (
    PROJECT_ROOT
    / "metrics"
    / "current_corpus_4793_anime"
    / f"svd_params_tuning_{n_runs}run_ndcg_2026.csv"
)
metrics_path.parent.mkdir(parents=True, exist_ok=True)
svd_component_summary.to_csv(metrics_path, index=False)

print(f"Saved {metrics_path.relative_to(PROJECT_ROOT)}")
svd_component_summary


Saved metrics\current_corpus_4793_anime\svd_params_tuning_100run_ndcg_2026.csv


,bayesian_uncertainty_weight,k,avg_precision_at_k,std_precision_at_k,avg_hit_rate,std_hit_rate,avg_hits,avg_ndcg_at_k,std_ndcg_at_k,avg_mrr_at_k,...,baseline_avg_precision_at_k,baseline_std_precision_at_k,baseline_avg_hit_rate,baseline_std_hit_rate,baseline_avg_hits,component,n_feature,clip_predictions,n_runs,svd_explained_variance
16,14,5,0.578,0.192580,0.090313,0.030091,2.89,0.678817,0.203216,0.962000,...,0.102,0.131794,0.015938,0.020593,0.51,400,4000,False,100,0.406873
22,14,5,0.578,0.196731,0.090313,0.030739,2.89,0.672038,0.199202,0.957833,...,0.102,0.131794,0.015938,0.020593,0.51,400,5000,False,100,0.372490
10,14,5,0.574,0.189960,0.089688,0.029681,2.87,0.686938,0.195747,0.968333,...,0.102,0.131794,0.015938,0.020593,0.51,400,3000,False,100,0.450673
14,14,5,0.574,0.187821,0.089688,0.029347,2.87,0.672227,0.202477,0.959500,...,0.102,0.131794,0.015938,0.020593,0.51,300,4000,False,100,0.340988
4,14,5,0.564,0.171458,0.088125,0.026790,2.82,0.660718,0.186413,0.947833,...,0.102,0.131794,0.015938,0.020593,0.51,400,2000,False,100,0.524841
20,14,5,0.562,0.185744,0.087813,0.029023,2.81,0.653546,0.207305,0.951167,...,0.102,0.131794,0.015938,0.020593,0.51,300,5000,False,100,0.311321
12,14,5,0.560,0.170561,0.087500,0.026650,2.80,0.641917,0.200302,0.939500,...,0.102,0.131794,0.015938,0.020593,0.51,200,4000,False,100,0.264971
18,14,5,0.560,0.168175,0.087500,0.026277,2.80,0.639350,0.200120,0.941167,...,0.102,0.131794,0.015938,0.020593,0.51,200,5000,False,100,0.240519
8,14,5,0.552,0.186667,0.086250,0.029167,2.76,0.654121,0.198880,0.956667,...,0.102,0.131794,0.015938,0.020593,0.51,300,3000,False,100,0.379371
2,14,5,0.550,0.180627,0.085938,0.028223,2.75,0.640538,0.195833,0.937500,...,0.102,0.131794,0.015938,0.020593,0.51,300,2000,False,100,0.444022


## Results

This 100-run sweep evaluated the current 4,793-anime corpus with `clip_predictions=False`, `bayesian_uncertainty_weight=14`, and `top_ks=[5, 10]`. Each TF-IDF/SVD configuration was scored with hit-rate metrics plus NDCG/MRR, and the full table is saved to `metrics/current_corpus_4793_anime/svd_params_tuning_100run_ndcg_2026.csv`.

| Read | Components | Max Features | Precision@5 | NDCG@5 | MRR@5 | Precision@10 | NDCG@10 | MRR@10 | Notes |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---|
| Best top-5 precision | 400 | 4000 | 0.5780 | 0.6788 | 0.9620 | 0.3970 | 0.5165 | 0.9620 | Best choice if short recommendation lists matter most. |
| Tied top-5 precision | 400 | 5000 | 0.5780 | 0.6720 | 0.9578 | 0.3920 | 0.5163 | 0.9595 | Same Precision@5 as 400/4000, but weaker ranking tie-breakers. |
| Best top-10 precision | 300 | 4000 | 0.5740 | 0.6722 | 0.9595 | 0.4100 | 0.5175 | 0.9595 | Best Precision@10, still strong at top-5. |
| Best NDCG/MRR balance | 400 | 3000 | 0.5740 | 0.6869 | 0.9683 | 0.4050 | 0.5241 | 0.9683 | Best ranking-quality tie-breaker across both cutoffs. |

Decision: use **400 components / 4000 max features** if optimizing primarily for short top-5 recommendation precision. If the goal is a more balanced ranking-quality default, **400 components / 3000 max features** is the cleaner choice because it is near the precision leaders and wins the NDCG/MRR tie-breakers. The old low-dimensional settings are safely worse on this corpus, so there is no need to retest 50 or 100 components unless the corpus changes substantially.
